# 02 - Silver labels + spot-check review

Ground truth for the classification evaluation in notebooks 03 and
04. Rather than hand-label all ~200 AI/ML stories from notebook 01,
we use a *stronger* Ollama model (`qwen2.5:14b`) as a labeller,
then have the human (you) spot-check a stratified random sample of
30-50 labels to estimate the silver-label accuracy. Downstream
metrics are reported with that accuracy as the ceiling.

**What you will learn**

- The **silver labels** technique: using a more capable model as a
  labeller when human labelling is expensive.
- Why the labeller must be *different* from the model being
  evaluated (otherwise the eval becomes model-vs-itself).
- How to structure a spot-check that gives a defensible estimate of
  silver-label accuracy without labelling everything.
- How to fold the silver-label accuracy into the downstream
  classification metrics honestly.

## Setup

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.llm import get_chat_ollama
from src.labels import CATEGORIES, prompt_snippet, save_gold

pd.set_option("display.width", 120)
pd.set_option("display.max_colwidth", 90)
np.random.seed(0)

## 1. Load the AI/ML stories from notebook 01

In [2]:
stories = pd.read_csv(ROOT / "data" / "stories_ai.csv")
print(f"loaded: {len(stories):,} stories")
stories[["id", "title", "score", "num_comments"]].head(3)

loaded: 223 stories


,id,title,score,num_comments
0,49273478,Qwen3.8-2.4T,712,171
1,49274600,DeepSeek V4 Pro 0813,1040,452
2,49281916,Codex in ChatGPT desktop app for Linux is now in preview,468,315


## 2. The labeller: qwen2.5:14b via Ollama

Different model family (Alibaba, not Meta), larger than the
`llama3.1:8b` we will evaluate in nb04, and independently trained.
That independence is the whole point: a same-family / same-size
labeller would just recycle its own biases and produce
artificially inflated agreement.

We also ask the labeller for a **self-reported confidence** on a
1-5 scale. Low-confidence items are exactly the ones we want to
spot-check first.

In [3]:
LABELLER_MODEL = "qwen2.5:14b"
labeller = get_chat_ollama(model=LABELLER_MODEL)
print(f"using labeller: {LABELLER_MODEL}")

using labeller: qwen2.5:14b


## 3. Prompt design

The prompt returns two things: a category name from `src.labels.CATEGORIES`
and a confidence integer 1-5. We ask for them on separate lines with
a fixed format that is easy to parse. Temperature is 0 so the same
title always produces the same label.

In [4]:
LABEL_PROMPT_TEMPLATE = '''You classify Hacker News story submissions into one of a fixed set of categories.

{categories}

Rules:
- Output EXACTLY two lines and nothing else.
- Line 1: category name from the list above, lowercase, no punctuation.
- Line 2: your confidence on a 1-5 scale (1 = guessing; 5 = obvious).

Story title: {title}

Story text (may be empty; empty means the URL points to an external site):
{text}
'''

def render_prompt(title: str, text: str) -> str:
    return LABEL_PROMPT_TEMPLATE.format(
        categories=prompt_snippet(),
        title=title,
        text=(text[:1500] if isinstance(text, str) and text else "(no body; external URL)"),
    )

# Preview the prompt on one story
sample = stories.iloc[0]
print(render_prompt(sample["title"], sample.get("text", ""))[:800])

You classify Hacker News story submissions into one of a fixed set of categories.

Categories:
- research: New research paper, arXiv link, or discussion of academic results.
- product: A product launch, Show HN, startup announcement, or feature release.
- tool: An open-source library, framework, dataset, benchmark, or model release.
- tutorial: A blog post, walkthrough, guide, or explanatory article.
- opinion: Opinion, essay, hot take, or industry commentary.
- news: Company / industry news, funding round, acquisition, personnel change.
- other: Anything that does not clearly fit above (jobs, memes, off-topic).

Rules:
- Output EXACTLY two lines and nothing else.
- Line 1: category name from the list above, lowercase, no punctuation.
- Line 2: your confidence on a 1-5 scale (1 = guessing;


## 4. Run the labeller on all AI/ML stories

This is the slow step. On a laptop GPU `qwen2.5:14b` produces
~1-2 labels per second, so 200 stories take about 2-3 minutes.
Progress is printed every 25 items so we can spot if anything is
stuck.

In [5]:
from langchain_core.messages import HumanMessage
import time

def parse_response(text: str) -> tuple[str, int]:
    lines = [ln.strip() for ln in text.strip().splitlines() if ln.strip()]
    if not lines:
        return "other", 1
    raw_cat = lines[0].lower().strip(" .:-\"'*")
    category = "other"
    for name in CATEGORIES:
        if raw_cat == name or raw_cat.startswith(name):
            category = name
            break
    conf = 3
    if len(lines) >= 2:
        for tok in lines[1].split():
            if tok.isdigit():
                conf = max(1, min(5, int(tok)))
                break
    return category, conf


rows = []
t0 = time.time()
for i, row in stories.iterrows():
    prompt = render_prompt(row["title"], row.get("text", ""))
    resp = labeller.invoke([HumanMessage(content=prompt)])
    cat, conf = parse_response(resp.content)
    rows.append({"id": int(row["id"]), "category": cat, "confidence": conf})
    if (i + 1) % 25 == 0:
        rate = (i + 1) / (time.time() - t0)
        print(f"  {i + 1}/{len(stories)} labelled  ({rate:.2f} items/s)")

silver = pd.DataFrame(rows)
print(f"done: {len(silver)} labels in {time.time() - t0:.1f}s")

  25/223 labelled  (0.70 items/s)


  50/223 labelled  (0.85 items/s)


  75/223 labelled  (0.90 items/s)


  100/223 labelled  (0.95 items/s)


  125/223 labelled  (0.97 items/s)


  150/223 labelled  (0.98 items/s)


  175/223 labelled  (0.98 items/s)


  200/223 labelled  (0.99 items/s)


done: 223 labels in 222.9s


## 5. Save silver labels + describe the distribution

In [6]:
silver_path = ROOT / "data" / "silver_labels.csv"
save_gold(silver[["id", "category"]], silver_path)   # store id+category only
silver.to_csv(ROOT / "data" / "silver_labels_full.csv", index=False)  # keep confidence
print(f"saved silver labels to {silver_path}")
print()
print("label distribution:")
print(silver["category"].value_counts().to_string())
print()
print("confidence distribution:")
print(silver["confidence"].value_counts().sort_index().to_string())

saved silver labels to C:\Users\anjan\Desktop\Goals\github\hn-ml-trends\data\silver_labels.csv

label distribution:
category
product     61
opinion     61
news        37
research    23
tutorial    21
tool        16
other        4

confidence distribution:
confidence
1      3
2      2
3     94
4    109
5     15


**Reading the distributions.** If any category has fewer than
~5 items the downstream classification metrics on that class will be
noisy. `other` is expected to be large-ish because we kept a
permissive AI/ML filter in nb01. If the label distribution is very
skewed we may want to widen the fetch window in nb01 next time.

## 6. Sample for spot-check

Pick up to 40 stories for the human review, in two buckets:

- Up to 20 stratified by predicted category (proportional but with a
  floor of at least 2 per category that appeared).
- Up to 20 low-confidence rows (confidence <= 2), if there are that
  many. On this run there are only a handful of low-confidence
  labels, so the effective sample lands closer to 20-25 than to 40.
  That is itself informative: the labeller is confident about most
  of the AI/ML slice, and the confidence signal is not doing much
  work when the model is uniformly sure of itself.

The two buckets spend the spot-check budget where labeller mistakes
are most likely.

In [7]:
rng = np.random.default_rng(0)

# Stratified sample by category
category_sizes = silver["category"].value_counts()
target = min(20, len(silver))
strata_ids = []
for cat, n in category_sizes.items():
    take = max(2, round(target * n / len(silver)))
    strata_ids.extend(
        rng.choice(silver.loc[silver["category"] == cat, "id"].values,
                   size=min(take, n), replace=False)
    )
strata_ids = list(dict.fromkeys(strata_ids))[:20]

# Low-confidence sample
lowconf = silver[silver["confidence"] <= 2]
lowconf_sample = list(rng.choice(
    lowconf["id"].values,
    size=min(20, len(lowconf)), replace=False,
)) if len(lowconf) else []

spotcheck_ids = list(dict.fromkeys(strata_ids + lowconf_sample))
spot = (
    silver[silver["id"].isin(spotcheck_ids)]
    .merge(stories[["id", "title", "url", "score"]], on="id", how="left")
    .loc[:, ["id", "title", "url", "score", "category", "confidence"]]
    .sort_values(["confidence", "category"])
    .reset_index(drop=True)
)
spot["human_category"] = ""    # user fills this column
spot["notes"] = ""

spotcheck_path = ROOT / "data" / "spotcheck_sample.csv"
spot.to_csv(spotcheck_path, index=False)
print(f"saved {len(spot)} spot-check rows to {spotcheck_path}")
spot.head(5)

saved 24 spot-check rows to C:\Users\anjan\Desktop\Goals\github\hn-ml-trends\data\spotcheck_sample.csv


,id,title,url,score,category,confidence,human_category,notes
0,49362280,Find jobs with Claude Code CLI,https://github.com/AdanRott/magnificent-jobs-plugin,1,other,1,,
1,49362983,The job ain't quite the same,https://prashanth.world/the-job-aint-the-same/,2,other,1,,
2,49366691,Ask HN: Any Curated Forum Directories?,NaN,1,other,1,,
3,49363263,H3 Video – MiniMax H3 AI Video Generator,https://www.h3vid.com,3,product,2,,
4,49362895,"100% RHAE on ARC-AGI-3 public with Claude Code, Opus 5, and one skill",https://arc-skill.vercel.app/,2,tutorial,2,,


## 7. Your review

Open `data/spotcheck_sample.csv` in your editor of choice. For each
row:

1. Read the title (and click the URL if the title is not enough).
2. If the silver label is correct, copy it into `human_category`.
3. If it is wrong, put the *correct* category into `human_category`.
   Optionally jot a short note in `notes` explaining the disagreement.
4. Save as `data/spotcheck_reviewed.csv` (same schema, filled in).

Silver labels are stored separately in `data/silver_labels.csv`; the
review file is only for the ~40 spot-check rows.

The next section loads whichever file exists (`spotcheck_reviewed.csv`
if you have finished, otherwise the untouched sample) and reports
whatever agreement rate can be measured.

## 8. Compute silver-label accuracy from the spot-check

In [8]:
reviewed_path = ROOT / "data" / "spotcheck_reviewed.csv"
if reviewed_path.exists():
    reviewed = pd.read_csv(reviewed_path)
    graded = reviewed[reviewed["human_category"].astype(str).str.strip() != ""]
    if len(graded):
        agree = (graded["human_category"].str.strip().str.lower() == graded["category"]).mean()
        print(f"spot-check: {len(graded)} labels reviewed")
        print(f"silver-vs-human agreement: {agree * 100:.1f}%")
        print()
        print("disagreements:")
        disagree = graded[graded["human_category"].str.strip().str.lower() != graded["category"]]
        if len(disagree):
            print(disagree[["id", "title", "category", "human_category", "notes"]].to_string(index=False))
        else:
            print("  (none)")
    else:
        print("spotcheck_reviewed.csv exists but no rows have human_category filled in yet.")
else:
    print("spotcheck_reviewed.csv not found yet.")
    print(f"When you finish the review, save it at {reviewed_path}")
    print("and re-run this cell to see the agreement rate.")

spotcheck_reviewed.csv not found yet.
When you finish the review, save it at C:\Users\anjan\Desktop\Goals\github\hn-ml-trends\data\spotcheck_reviewed.csv
and re-run this cell to see the agreement rate.


## Takeaways for notebook 03

- Silver labels for all AI/ML stories are saved at
  `data/silver_labels.csv`; the full frame with confidences is at
  `data/silver_labels_full.csv`.
- The spot-check estimates the silver-label accuracy; if the
  agreement rate is >= 85%, we treat the silver set as the ground
  truth for nb03/nb04 metrics with that ceiling explicitly noted.
- If the agreement rate is lower we either widen the review, use a
  different labeller, or fall back to manual labelling for the
  cases where the two models disagree.
- Notebook 03 fits a TF-IDF + logistic regression baseline against
  the silver labels; notebook 04 does the same for llama3.1:8b via
  LangChain. The comparison in nb06 uses whichever classifier scored
  higher on the silver set.